# 🎯 [AMM02_M09] AAR 기반 소모량 예측치와 실사격 결과 간 편차 분석 및 소모 계수 재보정
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/AMM02/amm02_m09_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 탄약·특수물자 (Level 3) |
| **미션 ID** | AMM02_M09 |
| **미션 주제** | AAR 기반 소모량 예측치와 실사격 결과 간 편차 분석 및 소모 계수 재보정 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
탄약 저장고는 공간이 한정되어 있고 품목마다 함께 둘 수 없는 제약이 있어, 앞으로 얼마나 소모되고 얼마나 자주 반출될지를 알아야 입고 위치와 재보급 순서를 합리적으로 정할 수 있다. 그러나 작전 단계별 고정 기준 소모율(CSR)만으로 계획하면 교전 강도가 낮은 날은 과잉 재고가 쌓이고 높은 날은 긴급 재보급이 반복되어 저장 공간과 작업 동선이 낭비된다. 본 프로젝트는 교육용 가상 부대·가상 품목·가상 저장고 자료를 코드로 만들어, 탄약·유류 소모율 데이터 정의와 전처리, CSR 기반 Baseline과 탐색적 분석, 적 저항·화력 계획을 반영한 소모량 예측, 보급 수준(DOS) 연계 재보급 산출, 작전 지도 대시보드와 설명가능성, 저장 배치 최적화와 군수참모 결재선(HITL), 7일 고강도 스트레스 시험, AAR 기반 소모 계수 재보정을 단계별로 구현한다. 최종 목표는 예측 소모량을 근거로 재보급 소요와 저장 위치를 함께 제안해 저장 효율과 작업 동선을 개선하고, 불출·대체 탄종·재배치는 군수참모가 근거를 확인해 결정하도록 지원하는 통합 시뮬레이터를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📁 2. 실습 산출물(Deliverables) 디렉터리 구조 및 안내

> **💡 [학습 안내] 산출물 디렉터리 아키텍처**  
> 본 과제는 단순 일회성 노트북 실행에 그치지 않고, **실제 군수 야전(군수참모처, 탄약창)에 즉시 도입 가능한 "완성형 국방 AI 시뮬레이터 도구"를 완성하는 실무 엔지니어링 과정**입니다. 각 미션의 **산출물(Deliverables)**에 명시된 디렉터리 경로는 실무 표준 소프트웨어 아키텍처에 해당합니다:

| 디렉토리 | 군 실무 소프트웨어 아키텍처 분류 | 주요 보관 산출물 | 관련 미션 |
| :--- | :--- | :--- | :---: |
| **`docs/`** | **국방 행정 및 기술 명세서** | 도메인 데이터 명세서(`docs/amm02_logistics_domain_spec.md`), 표준 운영 절차 기술 매뉴얼(`docs/amm02_operations_guide.md`) | 미션 1, 10 |
| **`src/etl/`** | **군수 데이터 정제 모듈** | 10종 정제 룰 및 긴급 불출 보존 ETL 파이프라인(`src/etl/amm02_data_cleansing.py`) | 미션 2 |
| **`notebooks/`** | **분석 및 통계 검증** | 90일 소모량 Baseline 비교, 가설 검정 및 ABC 파레토 분석(`notebooks/amm02_eda_analysis.ipynb`) | 미션 3 |
| **`src/features/`** | **전술 피처 엔지니어링** | 시계열 파생 피처 엔지니어링 모듈(`src/features/amm02_feature_engineering.py`) | 미션 4 |
| **`src/models/`** | **AI 예측 & 재보급 엔진** | GBR+RF 앙상블 및 DOS 재보급 산출 엔진(`src/models/amm02_advanced_model.py`) | 미션 5 |
| **`src/explainability/`** | **XAI 설명가능성 모듈** | XAI 기여도 분석 및 자연어 보고서 생성기(`src/explainability/amm02_xai_explainer.py`) | 미션 6 |
| **`src/optimization/`** | **저장 최적화 & HITL** | 정수계획 저장 최적화 및 참모 HITL 결재선 엔진(`src/optimization/amm02_simulator.py`) | 미션 7 |
| **`src/dashboard/`** | **작전 지도 대시보드** | 전술 격자 지도 연동 인터랙티브 대시보드 웹앱(`src/dashboard/amm02_app.py`) | 미션 8 |
| **`reports/`** | **검증 및 AAR 리포트** | 사후검토(AAR) 편차 분석, 축소 추정 CSR 재보정 및 3대 정량 효과 리포트(`reports/amm02_roi_analysis.md`) | 미션 9 |
| **`pipeline/`** | **MLOps 자동 재학습** | 신규 훈련 데이터 기반 챔피언/챌린저 모델 자동 성능 비교 및 롤오버 파이프라인(`pipeline/amm02_retrain.py`) | 미션 10 |

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 09. AAR 기반 소모량 예측치와 실사격 결과 간 편차 분석 및 소모 계수 재보정
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

AAR 기반 소모량 예측 편차를 분석하고 소모 계수를 재보정한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **AAR 자료로 CSR 계획치와 실소모 결과의 편차를 분석하고 소모 계수를 재보정해 도입 효과를 산출한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **작전 단계·품목·교전 강도 구간별로 실제/계획 비율을 분석해 편차 원인을 찾는다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `reports/amm02_roi_analysis.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 추정 산식의 객관성 및 군 재정/군수 기획 부서 설득력

#### 미션 PASS 조건

- 정량 기준: 비용 절감액, 시간 단축률 등 3대 정량 효과 지표 산출


- 진행 기준: ROI 분석 보고서 완성 및 주요 지표 민감도 분석 수록


- 검증 방법: 회계/기획 기준 모델과의 비교 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""AAR(사후검토) 자료로 기준 소모율(CSR) 계획치와 실사격·실소모 결과의 편차를 작전 단계·품목·교전 강도별로 분석해 원인을 찾고, 소모 계수를 축소 추정으로 재보정한 뒤 새 훈련 자료에서 검증하며, 비용 절감·작업시간 단축·긴급 재보급 감소 3대 효과 지표와 민감도를 산출한다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 부대·가상 품목 일일 소모 기록 생성기 (실제 탄약 제원·소모율과 무관)
# 가상 부대 8개 x 90일. 소모량 = 가상 CSR 계수 x 병력 규모 x (교전 강도·적 저항·화력 계획 효과) x 잡음
# ---------------------------------------------------------------------------
UNITS = [f"U{i:02d}" for i in range(1, 9)]
ITEMS = ["ammo_A", "ammo_B", "ammo_C", "ammo_D", "fuel_jet", "fuel_diesel"]
AMMO, FUEL = ITEMS[:4], ITEMS[4:]
PHASES = ["prep", "attack", "defense", "stability"]
CSR = {  # 가상 계수: 병력 100명·1일 기준 소모 단위(탄약=상자, 유류=드럼)
    "prep":      [1.0, 0.6, 0.3, 0.2, 1.5, 6.0],
    "attack":    [6.0, 3.5, 2.0, 1.2, 5.0, 12.0],
    "defense":   [4.5, 3.0, 1.6, 0.8, 3.0, 7.0],
    "stability": [0.8, 0.5, 0.3, 0.2, 1.0, 5.0],
}
PHASE_RESIST = {"prep": 1.0, "attack": 3.2, "defense": 3.6, "stability": 0.8}


def make_ops(n_days=90, seed=7, surge=0):
    """가상 부대별 일일 작전 기록과 품목별 실제 소모량(정답)을 만든다. surge>0이면 마지막 surge일을 고강도 공격 단계로 바꾼다(스트레스 시험용)."""
    rng = np.random.default_rng(seed)
    rows = []
    for uid in UNITS:
        size = int(rng.integers(3, 9)) * 100
        lens = rng.integers(8, 20, 4)
        sched = ["prep"] * lens[0] + ["attack"] * lens[1] + ["defense"] * lens[2] + ["attack"] * lens[3]
        sched = (sched + ["stability"] * n_days)[:n_days]
        if surge:
            sched[-surge:] = ["attack"] * surge
        r = 1.0
        for d, ph in enumerate(sched):
            r = float(np.clip(0.7 * r + 0.3 * (4.8 if surge and d >= n_days - surge else PHASE_RESIST[ph]) + rng.normal(0, 0.35), 0, 5))
            inten = float(np.clip(r / 5 * 0.9 + rng.normal(0, 0.06), 0, 1))
            fire = int(rng.poisson(1 + 8 * inten))
            move = float(rng.gamma(2, {"prep": 4, "attack": 12, "defense": 2, "stability": 5}[ph]))
            weather = str(rng.choice(["clear", "rain", "snow"], p=[0.7, 0.2, 0.1]))
            emerg = bool(rng.random() < 0.02)
            rec = {"unit": uid, "day": d, "phase": ph, "unit_size": size, "intensity": round(inten, 3),
                   "resistance": round(r, 2), "fire_missions": fire, "move_km": round(move, 1),
                   "weather": weather, "emergency": emerg}
            base = size / 100
            for i, it in enumerate(ITEMS):
                c = CSR[ph][i] * base
                if it in AMMO:
                    c *= (0.5 + 0.6 * inten) * (1 + 0.04 * fire * (1.0 if it in ("ammo_C", "ammo_D") else 0.4))
                elif it == "fuel_jet":
                    c *= (0.6 + 0.5 * inten) + 0.03 * fire
                else:
                    c = c * (0.8 + 0.2 * (weather != "clear")) + 0.05 * move * base
                c *= rng.lognormal(0, 0.12) * (2.5 if emerg else 1.0)
                rec[it] = round(float(c), 1)
            rows.append(rec)
    return pd.DataFrame(rows)
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 저장고(H1~H3) 저장칸 배치 모델. 실제 탄약 저장 기준·안전거리와 무관한 가상 계수다.
# 저장칸: 저장고 x 열(row) x 칸(col). 동선 거리 = 반출장(입구)까지의 가상 거리(m).
# 저장군(G1~G3)은 가상 혼적 제한 그룹이며, 저장고마다 받을 수 있는 저장군이 정해져 있다.
# ---------------------------------------------------------------------------
HOUSES = {"H1": {"groups": {"G1", "G2"}, "rows": 2, "cols": 6, "dock": 15.0},
          "H2": {"groups": {"G1"}, "rows": 2, "cols": 5, "dock": 40.0},
          "H3": {"groups": {"G2", "G3"}, "rows": 2, "cols": 6, "dock": 70.0}}
ITEM_GROUP = {"ammo_A": "G1", "ammo_B": "G1", "ammo_C": "G2", "ammo_D": "G3"}
SLOT_CAP = 6                      # 1열 저장칸 가상 팔레트 수용량(2열은 +4)


def make_slots(seed=3):
    """저장칸 목록과 현재 점유(이동 불가 기존 재고) 여부를 만든다."""
    rng = np.random.default_rng(seed)
    rows = []
    for h, c in HOUSES.items():
        for r in range(c["rows"]):
            for k in range(c["cols"]):
                rows.append({"slot": f"{h}-{r + 1}{k + 1:02d}", "house": h, "dist_m": c["dock"] + 6.0 * k + 4.0 * r,
                             "cap": SLOT_CAP if r == 0 else SLOT_CAP + 4, "fixed": bool(rng.random() < 0.15)})
    return pd.DataFrame(rows)


def make_skus(demand, seed=5):
    """품목별 예측 일일 반출량(demand: 품목->상자/일)을 로트 단위 SKU로 나눈다. 반출 빈도 = 일일 반출 횟수(가상)."""
    rng = np.random.default_rng(seed)
    rows = []
    for it, dq in demand.items():
        n_lot = 5
        share = rng.dirichlet(np.ones(n_lot) * 2)
        for j in range(n_lot):
            rows.append({"sku": f"{it}-L{j + 1}", "item": it, "group": ITEM_GROUP[it],
                         "pallets": int(rng.integers(2, SLOT_CAP + 5)), "freq": round(float(dq * share[j] / 4), 2)})
    return pd.DataFrame(rows)


def feasible(sku, slot):
    return (sku["group"] in HOUSES[slot["house"]]["groups"]) and sku["pallets"] <= slot["cap"] and not slot["fixed"]


def travel(assign, skus, slots):
    """작업동선 = 반출 빈도 x 왕복 거리 합(가상 m/일)."""
    d = slots.set_index("slot")["dist_m"]
    return float(sum(2 * skus.loc[i, "freq"] * d[s] for i, s in assign.items()))


def first_fit(skus, slots):
    """현행 방식(대안 비교 기준): 입고 순서대로 비어 있는 첫 번째 적합 칸에 넣는다."""
    used, out = set(), {}
    for i, k in skus.iterrows():
        for _, s in slots.iterrows():
            if s["slot"] not in used and feasible(k, s):
                out[i] = s["slot"]; used.add(s["slot"]); break
    return out


def optimize(skus, slots, lam=8):
    """할당 문제(정수계획 특수형): 비용 = 반출 빈도 x 왕복거리 + lam x 빈 공간(공간낭비). 부적합 칸은 큰 비용."""
    cost = np.full((len(skus), len(slots)), 1e7)
    for i, k in skus.iterrows():
        for j, s in slots.iterrows():
            if feasible(k, s):
                cost[i, j] = 2 * k["freq"] * s["dist_m"] + lam * (s["cap"] - k["pallets"])
    ri, cj = linear_sum_assignment(cost)
    return {int(i): slots.loc[j, "slot"] for i, j in zip(ri, cj) if cost[i, j] < 1e7}, cost


def loc_score(assign, skus, slots):
    """위치점수(0~100): 반출 빈도 상위 SKU일수록 반출장 가까운 칸에 있는 정도(가중 평균)."""
    d = slots.set_index("slot")["dist_m"]; dmin, dmax = d.min(), d.max()
    w = skus["freq"] / skus["freq"].sum()
    return float(100 * sum(w[i] * (1 - (d[s] - dmin) / (dmax - dmin)) for i, s in assign.items()))
# ---------------------------------------------------------------------------

PRICE = {"hold": 1.0, "emerg": 25.0}   # [교육용가상자료] 과잉 1단위 보관·수송 비용 지수, 긴급 재보급 1회 비용 지수
SHRINK_K = 20                            # 재보정 축소 강도(표본 n이 작을수록 기존 계수 유지)
SPEED = 80.0                             # 지게차 가상 이동 속도(m/분)


def long_ratio(ops):
    """(부대, 일자, 품목)별 실제 소모량과 CSR 계획량, 비율."""
    out = []
    for i, it in enumerate(ITEMS):
        plan = np.array([CSR[p][i] for p in ops["phase"]]) * ops["unit_size"] / 100
        out.append(pd.DataFrame({"unit": ops["unit"], "day": ops["day"], "phase": ops["phase"], "item": it,
                                 "intensity": ops["intensity"], "plan": plan, "qty": ops[it], "emergency": ops["emergency"]}))
    d = pd.concat(out, ignore_index=True)
    return d.assign(ratio=d["qty"] / d["plan"])


aar = long_ratio(make_ops(90, seed=7))          # 지난 훈련(AAR 대상)
new = long_ratio(make_ops(90, seed=8))          # 다음 훈련(재보정 검증용, 같은 생성 규칙의 다른 가상 부대)

# 1) 편차 분석: 단계 x 품목별 실제/계획 비율 중앙값(1보다 크면 계획 과소)
piv = aar[~aar["emergency"]].groupby(["phase", "item"])["ratio"].median().unstack()[ITEMS]
print("[AAR 편차: 실제/계획 비율 중앙값(긴급 불출 제외)]")
print(piv.to_string(float_format=lambda x: f"{x:.2f}"))
band = pd.cut(aar["intensity"], [0, 0.3, 0.6, 1.0], labels=["low", "mid", "high"], include_lowest=True)
amm = aar[aar["item"].isin(AMMO)].groupby(band, observed=True)["ratio"].median()
print("탄약 비율 vs 교전 강도:", ", ".join(f"{k} {v:.2f}" for k, v in amm.items()),
      "-> 원인: CSR은 단계별 고정값이라 같은 공격 단계라도 강도가 낮은 날은 과대, 높은 날은 과소 계획된다")
em_share = aar.loc[aar["emergency"], "qty"].sum() / aar["qty"].sum()
print(f"긴급 불출 비중 {em_share:.1%} -> 계수 재보정에서 제외하고 예비 재고로 관리")

# 2) 재보정: 단계 x 품목 x 계획 교전 강도 구간별 보정 계수 = 비율 중앙값^(n/(n+K)) (표본이 적은 칸은 기존 계수 유지 쪽으로 축소)
BANDS = [0, 0.3, 0.6, 1.0]
for d in (aar, new):
    d["band"] = pd.cut(d["intensity"], BANDS, labels=["low", "mid", "high"], include_lowest=True).astype(str)
base = aar[~aar["emergency"]]
g = base.groupby(["phase", "item", "band"])["ratio"]
FACT = (g.median() ** (g.size() / (g.size() + SHRINK_K))).to_dict()
print(f"재보정 계수 {len(FACT)}개(단계 4 x 품목 6 x 강도 구간), 예: attack-ammo_A low/mid/high = "
      + "/".join(f"{FACT.get(('attack', 'ammo_A', b), 1):.2f}" for b in ("low", "mid", "high")))


def mape(y, p):
    return float(np.mean(np.abs(p - y) / y))


def apply_csr(d):
    """재보정 계획량 = 기존 CSR 계획량 x 보정 계수(해당 칸이 없으면 1.0)."""
    return d["plan"] * np.array([FACT.get(k, 1.0) for k in zip(d["phase"], d["item"], d["band"])])


aar["plan2"], new["plan2"] = apply_csr(aar), apply_csr(new)


m1, m2 = mape(new["qty"], new["plan"]), mape(new["qty"], new["plan2"])
print(f"\n[재보정 검증(다음 훈련 자료)] 일일 MAPE 기존 CSR {m1:.1%} -> 재보정 {m2:.1%} ({m1 - m2:+.1%}p 개선)")
tot = new.groupby("item")[["qty", "plan", "plan2"]].sum()
tot["기존 오차"], tot["재보정 오차"] = tot["plan"] / tot["qty"] - 1, tot["plan2"] / tot["qty"] - 1
print("품목별 총량 오차:", ", ".join(f"{k} {a:+.1%}->{b:+.1%}" for k, a, b in zip(tot.index, tot["기존 오차"], tot["재보정 오차"])))


# 3) 3대 효과 지표(가상 단가, 90일 x 부대 8 기준). 두 계획 모두 AAR 자료에서 95% 충족 안전계수를 정해 같은 서비스 수준으로 비교한다
SAFE = {c: float(np.quantile(aar.loc[~aar["emergency"], "qty"] / aar.loc[~aar["emergency"], c], 0.95)) for c in ("plan", "plan2")}
print(f"95% 충족 안전계수: 기존 x{SAFE['plan']:.2f}, 재보정 x{SAFE['plan2']:.2f} -> 같은 충족 수준에서 공급 계획량을 비교한다")


def effect(plan_col, p_hold=PRICE["hold"], p_em=PRICE["emerg"]):
    supply = new[plan_col] * SAFE[plan_col]
    excess = (supply - new["qty"]).clip(lower=0).sum()
    short = int((new["qty"] > supply).sum())          # 공급 계획 초과 소모 -> 긴급 재보급 1회
    return excess * p_hold + short * p_em, short


c1, s1 = effect("plan"); c2, s2 = effect("plan2")
ops = make_ops(90, seed=7)
demand = ops[ops["day"] >= 76][AMMO].sum().div(14).to_dict()
skus, slots = make_skus(demand), make_slots()
t_cur, t_opt = travel(first_fit(skus, slots), skus, slots), travel(optimize(skus, slots)[0], skus, slots)
kpis = pd.DataFrame([["비용(과잉 보관·수송+긴급 재보급) 지수", c1, c2, 1 - c2 / c1],
                     ["저장고 반출 작업시간(분/일)", t_cur / SPEED, t_opt / SPEED, 1 - t_opt / t_cur],
                     ["긴급 재보급 횟수(회/90일)", s1, s2, 1 - s2 / max(s1, 1)]], columns=["지표", "현행", "개선", "개선율"])
print("\n[3대 정량 효과 지표(가상 단가 기준 상대 비교값)]")
print(kpis.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
print(f"효과 지표 산출 {len(kpis)}종 (목표 3대 지표: {'달성' if len(kpis) >= 3 else '미달'}) - 긴급 재보급 횟수는 같은 충족 수준에서 비슷하게 유지되는지(차이 {s2 - s1:+d}회) 확인하는 지표다")

sens = []
for ph in (0.8, 1.0, 1.2):
    for pe in (15.0, 25.0, 40.0):
        a, _ = effect("plan", PRICE["hold"] * ph, pe); b, _ = effect("plan2", PRICE["hold"] * ph, pe)
        sens.append((ph, pe, 1 - b / a))
s = pd.DataFrame(sens, columns=["보관단가 배수", "긴급 1회 비용", "비용 절감률"]).pivot(index="보관단가 배수", columns="긴급 1회 비용", values="비용 절감률")
print("\n[민감도: 비용 절감률]")
print(s.to_string(float_format=lambda x: f"{x:.1%}"))
print("도입 로드맵: 단기(1~3개월) 훈련 AAR로 계수 재보정 / 중기(4~9개월) 예측 모델·저장 배치 최적화 시범 운용 / 장기(10개월~) 군수 정보체계 연계와 정기 재학습")
print("금액·시간은 가정 단가로 계산한 상대 비교값이며 예산 근거로 쓰지 않는다.")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
im = ax[0].imshow(piv.values, cmap="coolwarm", vmin=0.5, vmax=1.5); ax[0].set_xticks(range(6)); ax[0].set_xticklabels(ITEMS, rotation=30)
ax[0].set_yticks(range(4)); ax[0].set_yticklabels(piv.index); ax[0].set_title("AAR actual/plan ratio (median)"); plt.colorbar(im, ax=ax[0])
x = np.arange(len(tot)); ax[1].bar(x - 0.2, tot["기존 오차"], 0.4, label="old CSR"); ax[1].bar(x + 0.2, tot["재보정 오차"], 0.4, label="recalibrated")
ax[1].set_xticks(x); ax[1].set_xticklabels(tot.index, rotation=30); ax[1].axhline(0, c="k", lw=0.8); ax[1].set_title("Total plan error on next exercise"); ax[1].legend()
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 09 예상 실행 결과 (Output)</b></summary>

```text
[AAR 편차: 실제/계획 비율 중앙값(긴급 불출 제외)]
item       ammo_A  ammo_B  ammo_C  ammo_D  fuel_jet  fuel_diesel
phase                                                           
attack       0.90    0.89    0.99    1.02      1.04         0.94
defense      0.96    0.99    1.05    1.09      1.10         0.85
prep         0.61    0.64    0.62    0.67      0.76         0.90
stability    0.62    0.63    0.67    0.64      0.76         0.94
탄약 비율 vs 교전 강도: low 0.62, mid 0.90, high 1.06 -> 원인: CSR은 단계별 고정값이라 같은 공격 단계라도 강도가 낮은 날은 과대, 높은 날은 과소 계획된다
긴급 불출 비중 6.0% -> 계수 재보정에서 제외하고 예비 재고로 관리
재보정 계수 60개(단계 4 x 품목 6 x 강도 구간), 예: attack-ammo_A low/mid/high = 0.89/0.87/1.02

[재보정 검증(다음 훈련 자료)] 일일 MAPE 기존 CSR 34.0% -> 재보정 14.5% (+19.5%p 개선)
품목별 총량 오차: ammo_A +12.3%->+0.2%, ammo_B +11.6%->+0.2%, ammo_C -0.4%->-4.4%, ammo_D +0.9%->-3.4%, fuel_diesel +2.1%->-4.0%, fuel_jet +0.1%->-2.7%
95% 충족 안전계수: 기존 x1.23, 재보정 x1.26 -> 같은 충족 수준에서 공급 계획량을 비교한다

[3대 정량 효과 지표(가상 단가 기준 상대 비교값)]
                    지표       현행       개선   개선율
비용(과잉 보관·수송+긴급 재보급) 지수 21556.08 19063.11  0.12
      저장고 반출 작업시간(분/일)    18.88    14.97  0.21
      긴급 재보급 횟수(회/90일)   283.00   284.00 -0.00
효과 지표 산출 3종 (목표 3대 지표: 달성) - 긴급 재보급 횟수는 같은 충족 수준에서 비슷하게 유지되는지(차이 +1회) 확인하는 지표다

[민감도: 비용 절감률]
긴급 1회 비용  15.0  25.0  40.0
보관단가 배수                   
0.8      12.6% 10.7%  8.6%
1.0      13.4% 11.6%  9.6%
1.2      13.9% 12.3% 10.4%
도입 로드맵: 단기(1~3개월) 훈련 AAR로 계수 재보정 / 중기(4~9개월) 예측 모델·저장 배치 최적화 시범 운용 / 장기(10개월~) 군수 정보체계 연계와 정기 재학습
금액·시간은 가정 단가로 계산한 상대 비교값이며 예산 근거로 쓰지 않는다.
```
</details>
